# 🔑 Solutions: Lesson 13 · Transformers & Attention

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 13.1: Scaled dot-product attention with a mask  &nbsp; `⭐⭐`

Implement `attention(Q, K, V, mask=None)` returning `(output, weights)`. Scores = `Q Kᵀ / √d_k`; where `mask` is `False` the score must become a huge negative number (e.g. −1e9) *before* the softmax; softmax over the last axis (keys).


In [2]:
def attention(Q, K, V, mask=None):
    scores = Q @ K.T / np.sqrt(K.shape[-1])
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    e = np.exp(scores - scores.max(-1, keepdims=True))
    weights = e / e.sum(-1, keepdims=True)
    return weights @ V, weights

In [3]:
# ▶ self-check
X = np.array([[1., 0.], [0., 1.], [1., 1.]])
r = attention(X, X, X)
dl.check("worked example, row 1 weights (.40, .20, .40)", r is not None and np.allclose(r[1][0], [.40, .20, .40], atol=.01))
dl.check("worked example, row 1 output (.80, .60)", r is not None and np.allclose(r[0][0], [.80, .60], atol=.01))
dl.check("rows of weights sum to 1", r is not None and np.allclose(r[1].sum(1), 1))
rc = attention(X, X, X, mask=np.tril(np.ones((3, 3), bool)))
dl.check("causal mask: token 0 attends only to itself", rc is not None and np.allclose(rc[1][0], [1, 0, 0], atol=1e-6))
dl.check("causal mask: token 1 ignores token 2", rc is not None and rc[1][1, 2] < 1e-6)

✅ worked example, row 1 weights (.40, .20, .40)
✅ worked example, row 1 output (.80, .60)
✅ rows of weights sum to 1
✅ causal mask: token 0 attends only to itself
✅ causal mask: token 1 ignores token 2


---
### 🧩 Problem 13.2: Sinusoidal positional encoding  &nbsp; `⭐⭐`

Write `sinusoidal_pe(n_pos, d)` (d even) returning an `(n_pos, d)` array with `PE[pos, 2i] = sin(pos / 10000^(2i/d))` and `PE[pos, 2i+1] = cos(...)`.


In [4]:
def sinusoidal_pe(n_pos, d):
    pos = np.arange(n_pos)[:, None]; i2 = np.arange(0, d, 2)[None, :]
    angle = pos / 10000 ** (i2 / d)
    pe = np.zeros((n_pos, d)); pe[:, 0::2], pe[:, 1::2] = np.sin(angle), np.cos(angle)
    return pe

In [5]:
# ▶ self-check
pe = sinusoidal_pe(10, 8)
dl.check("shape (10, 8)", pe is not None and pe.shape == (10, 8))
dl.check("position 0 is [0,1,0,1,...]", pe is not None and np.allclose(pe[0], [0, 1] * 4))
dl.check("PE[1,0] = sin(1) and PE[1,1] = cos(1)", pe is not None and np.isclose(pe[1, 0], np.sin(1)) and np.isclose(pe[1, 1], np.cos(1)))
dl.check("last pair uses a much slower wave: PE[1,6] = sin(1/10000^(6/8))", pe is not None and np.isclose(pe[1, 6], np.sin(1 / 10000 ** 0.75)))

✅ shape (10, 8)
✅ position 0 is [0,1,0,1,...]
✅ PE[1,0] = sin(1) and PE[1,1] = cos(1)
✅ last pair uses a much slower wave: PE[1,6] = sin(1/10000^(6/8))


---
### 🧩 Problem 13.3: Count a Transformer block  &nbsp; `⭐⭐`

Write `block_params(d, d_ff)`: the number of parameters in one encoder block: attention (`Q,K,V` and output projections, each with bias), the two feed-forward linear layers (with biases), and two LayerNorms (weight + bias each). Verify against PyTorch's `nn.TransformerEncoderLayer`.


In [6]:
def block_params(d, d_ff):
    return (4 * d * d + 4 * d) + (d * d_ff + d_ff) + (d_ff * d + d) + 2 * (2 * d)

In [7]:
# ▶ self-check
ref = nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=256, batch_first=True, norm_first=True)
dl.check("matches nn.TransformerEncoderLayer (d=64, d_ff=256)", block_params(64, 256) == dl.count_params(ref))
dl.check("also for d=32, d_ff=64", block_params(32, 64) == dl.count_params(nn.TransformerEncoderLayer(32, 4, 64, batch_first=True)))

✅ matches nn.TransformerEncoderLayer (d=64, d_ff=256)
✅ also for d=32, d_ff=64


**Why this works.** Roughly 12·d² parameters per block (4d² in attention + 8d² in the feed-forward when d_ff = 4d). That is why Transformer size scales with the *square* of the model width.

---
### 🧩 Problem 13.4: Split and merge heads  &nbsp; `⭐⭐`

Write `split_heads(x, h)` turning `(B, T, d)` into `(B, h, T, d/h)`, and `merge_heads(x)` as its exact inverse. These two reshapes are the core of multi-head attention.


In [8]:
def split_heads(x, h):
    B, T, d = x.shape
    return x.reshape(B, T, h, d // h).transpose(1, 2)

def merge_heads(x):
    B, h, T, dk = x.shape
    return x.transpose(1, 2).reshape(B, T, h * dk)

In [9]:
# ▶ self-check
x = torch.randn(2, 5, 12)
s = split_heads(x, 3)
dl.check("split shape (2, 3, 5, 4)", s is not None and tuple(s.shape) == (2, 3, 5, 4))
dl.check("head 0 holds the first 4 features of every token", s is not None and torch.equal(s[:, 0], x[:, :, :4]))
dl.check("merge(split(x)) == x", s is not None and torch.equal(merge_heads(s), x))

✅ split shape (2, 3, 5, 4)
✅ head 0 holds the first 4 features of every token
✅ merge(split(x)) == x


---
### 🧩 Problem 13.5: Patchify an image (challenge)  &nbsp; `⭐⭐⭐`

Write `patchify(x, p)`: `(B, C, H, W) → (B, (H/p)·(W/p), C·p·p)` where patches are taken in **row-major** order and each patch is flattened. Use only reshape/permute (no loops).


In [10]:
def patchify(x, p):
    B, C, H, W = x.shape
    return x.reshape(B, C, H // p, p, W // p, p).permute(0, 2, 4, 1, 3, 5).reshape(B, (H // p) * (W // p), C * p * p)

In [11]:
# ▶ self-check
x = torch.arange(16, dtype=torch.float32).reshape(1, 1, 4, 4)          # [[0 1 2 3],[4 5 6 7],[8 9 10 11],[12 13 14 15]]
r = patchify(x, 2)
dl.check("shape (1, 4, 4)", r is not None and tuple(r.shape) == (1, 4, 4))
dl.check("first patch = top-left 2×2 = [0, 1, 4, 5]", r is not None and r[0, 0].tolist() == [0, 1, 4, 5])
dl.check("second patch = top-right = [2, 3, 6, 7]", r is not None and r[0, 1].tolist() == [2, 3, 6, 7])
dl.check("last patch = bottom-right = [10, 11, 14, 15]", r is not None and r[0, 3].tolist() == [10, 11, 14, 15])
dl.check("works for 28×28 / 7 → 16 patches of 49", patchify(torch.randn(3, 1, 28, 28), 7).shape == (3, 16, 49))

✅ shape (1, 4, 4)
✅ first patch = top-left 2×2 = [0, 1, 4, 5]
✅ second patch = top-right = [2, 3, 6, 7]
✅ last patch = bottom-right = [10, 11, 14, 15]
✅ works for 28×28 / 7 → 16 patches of 49
